# Sentimind: Run Full Pipeline (One Pass)
Notebook nay chay lai toan bo pipeline: preprocess -> train BiLSTM -> eval, sau do doc metrics cuoi.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Python = {sys.executable}")

ARTIFACTS_DIR = PROJECT_ROOT / "data" / "artifacts"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PRE = PROJECT_ROOT / "configs" / "preprocessing.yaml"
CONFIG_BILSTM = PROJECT_ROOT / "configs" / "bilstm.yaml"

for p in [ARTIFACTS_DIR, PROCESSED_DIR, CONFIG_PRE, CONFIG_BILSTM]:
    print("-", p)

PROJECT_ROOT = /home/sakana/Code/PTIT/NLP/sentimind
Python = /home/sakana/miniconda3/bin/python
- /home/sakana/Code/PTIT/NLP/sentimind/data/artifacts
- /home/sakana/Code/PTIT/NLP/sentimind/data/processed
- /home/sakana/Code/PTIT/NLP/sentimind/configs/preprocessing.yaml
- /home/sakana/Code/PTIT/NLP/sentimind/configs/bilstm.yaml


In [2]:
def run_cmd(cmd: list[str], cwd: Path):
    print("\n$ " + " ".join(cmd))
    proc = subprocess.run(
        cmd,
        cwd=str(cwd),
        text=True,
        capture_output=True,
        check=False,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with code {proc.returncode}: {' '.join(cmd)}")

print("Helper ready.")

Helper ready.


## 1) Preprocess
Sinh lai train/val/test va preprocessing_report.json.

In [3]:
run_cmd(["python", "scripts/preprocess.py", "--config", str(CONFIG_PRE)], PROJECT_ROOT)


$ python scripts/preprocess.py --config /home/sakana/Code/PTIT/NLP/sentimind/configs/preprocessing.yaml
11:49:53 | INFO     | __main__ | Loading raw data from data/raw/Combined Data.csv …
11:49:53 | INFO     | __main__ | Raw shape: (53043, 3)
11:49:53 | INFO     | src.data.preprocess | Dropped 362 null rows.
11:49:54 | INFO     | src.data.preprocess | Dropped 6 rows with text shorter than 3 chars after cleaning.
11:49:54 | INFO     | src.data.preprocess | Dropped 1630 duplicate (text, label) rows.
11:49:54 | INFO     | src.data.preprocess | Preprocessing done. 51045 / 53043 rows retained. Class distribution: {0: 16007, 1: 15093, 2: 3620, 3: 2501, 4: 894, 5: 2292, 6: 10638}
11:49:54 | INFO     | __main__ | Split sizes — train: 35731 | val: 7657 | test: 7657
11:49:55 | INFO     | __main__ | Splits saved to data/processed.
11:49:55 | INFO     | __main__ | Quality report saved to data/artifacts/preprocessing_report.json.
11:49:55 | INFO     | __main__ | ✓ All split files pass schema valid

## 2) Train BiLSTM
Cell nay co the chay lau tuy theo CPU/GPU.

In [ ]:
run_cmd(["python", "scripts/train_bilstm.py", "--config", str(CONFIG_BILSTM)], PROJECT_ROOT)


$ python scripts/train_bilstm.py --config /home/sakana/Code/PTIT/NLP/sentimind/configs/bilstm.yaml


## 3) Evaluate
Danh gia tren test split va sinh bilstm_metrics.json + confusion matrix.

In [ ]:
run_cmd(["python", "scripts/eval_bilstm.py", "--config", str(CONFIG_BILSTM), "--split", "test"], PROJECT_ROOT)

## 4) Summary Metrics
Doc metrics de xem nhanh ket qua lan chay moi nhat.

In [ ]:
metrics_path = ARTIFACTS_DIR / "bilstm_metrics.json"
if not metrics_path.exists():
    raise FileNotFoundError(metrics_path)

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

print("Model:", metrics.get("model"))
print("Split:", metrics.get("split"))
print("Accuracy:", metrics.get("accuracy"))
print("Macro F1:", metrics.get("macro_f1"))
print("Weighted F1:", metrics.get("weighted_f1"))

print("\nPer-class F1:")
for label, score in metrics.get("per_class", {}).items():
    print(f"- {label}: F1={score['f1']:.4f}, support={score['support']}")